# TP2 - Market Basket Analysis (Partie 2)

INF8111 - Fouille de données, Automne 2026

### Membres de l'équipe
    - Lhuisset, Maxence - 2124437
    - Guindon, Théo - 2255253

In [1]:
ROOT = "gs://tp2-inf8111-theo/notebooks/jupyter/data/"

In [2]:
import pyspark
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
df = spark.sql("select 'spark' as hello ")
df.show()

+-----+
|hello|
+-----+
|spark|
+-----+



## 4. MBA pour le dataset complet (10 points)

Comme vous l'avez probablement remarqué, même pour un ensemble de données moins volumineux, l'algorithme MBA est coûteux en calcul. Pour cette raison, cette fois-ci, nous allons répéter le processus, en utilisant Google Cloud Platform (GCP) pour créer un grand cluster. Toutes les instructions pour créer un cluster avec Spark et soumettre un travail seront expliquées lors du laboratoire. Dans tous les cas, vous devez lire les instructions données dans le document ``Instruction_GCP.pdf``.

Cette fois, nous travaillerons avec le fichier ``OBSERVATIONS.csv``, qui contient plusieurs millions d'observations.

En utilisant les lignes de ``OBSERVATIONS.csv``, créez un bloc de données où chaque ligne contient la liste des symptômes observés pour chaque patient, de manière similaire au GroceryDataset. Ensuite, exécutez l'algorithme MBA pour cet ensemble de transactions.
Le format devra permettre d'appeler l'ensemble des méthodes précédemment implémentées. (Vous pouvez réutiliser votre code de la section 3).

- Vous devez signaler le temps passé pour effectuer cette tâche.
- La sortie doit contenir le nom des symptômes.
- **Important** : la première colonne (patterns) doit être classée par ordre alphabétique.
- Afin d'accélérer l'exécution, vous pouvez garder ce second notebook uniquement dédié à cette partie et l'insérer dans le rendu final.

**Remarque importante : joignez des captures d'écran de votre sortie et de votre configuration de cluster.**

**ATTENTION : Dès que le cluster sera configuré, GCP commencera à utiliser les 50 $ de crédits disponibles. Veuillez donc supprimer le cluster ainsi que le bucket de stockage associé une fois l’exécution terminée, afin d’éviter toute consommation inutile de crédits. Le simple fait d’arrêter le cluster ne suffit pas à empêcher la consommation de crédits. Les ressources créées doivent être supprimées.**


In [3]:
%%time
"""
TODO: create a query to create and struct the transactions from the OBSERVATIONS.csv file
"""
import pyspark.sql.functions as f

# cette fois on prend le gros fichier OBSERVATIONS.csv au complet (pas l'extract)
df_symptoms = spark.read.csv(ROOT + 'SYMPTOMS.csv', header=True, sep=',', inferSchema=True)
df_observations = spark.read.csv(ROOT + 'OBSERVATIONS.csv', header=True, sep=',', inferSchema=True)

df_observations.createOrReplaceTempView("OBSERVATIONS")
df_symptoms.createOrReplaceTempView("SYMPTOMS")

# meme requete que dans la partie 1 mais sans DATA (on garde juste les lignes de OBSERVATIONS)
# -> une ligne par patient avec la liste des noms de ses symptomes
# collect_set plutot que collect_list pour pas avoir 2 fois le meme symptome pour un patient
# on garde une liste et pas une string séparée par des virgules comme le GroceryDataset,
# parce que y'a des noms de symptomes qui contiennent des virgules ("Hallucination, visual" par ex)
transactions = spark.sql("""
    SELECT observ.VAERS_ID,
        COLLECT_SET(symp.SYMPTOM_NAME) AS transaction
    FROM OBSERVATIONS observ
    INNER JOIN SYMPTOMS symp ON observ.SYMPTOM_ID = symp.SYMPTOM_ID
    GROUP BY observ.VAERS_ID
""")
# on va la reutiliser juste apres donc on la garde en memoire
transactions.cache() 
print("Nombre de transactions (patients):", transactions.count())
transactions.show(5, truncate=False)

26/11/11 00:10:07 WARN SimpleTableFunctionRegistry: The function read_files replaced a previously registered function.
26/11/11 00:10:07 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.


Nombre de transactions (patients): 2599720
+--------+-----------------------------------+
|VAERS_ID|transaction                        |
+--------+-----------------------------------+
|27760   |[Rash, Chills, Oedema, Pyrexia]    |
|27974   |[Face oedema, Skin discolouration] |
|28024   |[Rash, Pruritus]                   |
|28836   |[Headache, Injection site reaction]|
|29834   |[Somnolence, Pyrexia]              |
+--------+-----------------------------------+
only showing top 5 rows

CPU times: user 38.6 ms, sys: 3.58 ms, total: 42.2 ms
Wall time: 10.6 s


### Empreinte carbonne

In [4]:
%%time
"""
TODO: run the MBA algorithm and print the requested output
"""
from itertools import combinations

# fonctions reprises de la partie 1 

def format_tuples(pattern):
    # affichage
    return (str(pattern[0]), str(pattern[1]))

def map_to_patterns(row):
    produits = row.transaction
    # pour le grocery dataset c'etait une string, ici c'est deja une liste
    if isinstance(produits, str):
        produits = produits.split(',')
    # on trie pour que (a,b) et (b,a) soient la meme cle, et set() au cas ou y'a des doublons
    produits = sorted(set(produits))
    #singletons, paires et trios
    for k in range(1, 4):
        for combi in combinations(produits, k):
            yield (combi, 1)

def reduce_patterns(v1, v2):
    return v1 + v2

def map_to_subpatterns(pattern):
    cle, nb = pattern
    # le (None, nb) c'est pour se rappeler combien de fois la cle elle-meme est apparue
    yield (cle, (None, nb))
    if len(cle) > 1:
        # enlever un produit a la fois
        for i in range(len(cle)):
            sous_cle = cle[:i] + cle[i+1:]
            yield (sous_cle, (cle[i], nb))

def map_to_assoc_rules(rule):
    cle, regles = rule
    # trouver le nombre total d'apparitions de la cle (celui avec None)
    total = 0
    for produit, nb in regles:
        if produit is None:
            total = nb
    # confiance = nb de fois avec la cle / nb de fois que la cle apparait
    assoc = [(produit, nb / total) for produit, nb in regles if produit is not None]
    return (cle, assoc)


patterns_rdd = transactions.rdd.flatMap(map_to_patterns)
combined_patterns_rdd = patterns_rdd.reduceByKey(reduce_patterns)
subpatterns_rdd = combined_patterns_rdd.flatMap(map_to_subpatterns)
combined_rules = subpatterns_rdd.groupByKey().mapValues(list)
assoc_rules = combined_rules.map(map_to_assoc_rules)

# cache parce qu'on s'en ressert pour le symptome 11320 apres (sinon ca recalcule tout)
assoc_rules.cache()

df_assoc = assoc_rules.map(format_tuples).toDF(['patterns', 'association_rules']).sort(['patterns'])
# truncate pour les symptomes frequents puisque la liste des regles est énorme et ca fait
# planter l'affichage du notebook
df_assoc.show(30, truncate=150)
print("Nombre de patterns:", assoc_rules.count())

+--------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------+
|                                                                                                patterns|                                                                                                                                     association_rules|
+--------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------+
|                                                          ("5'nucleotidase increased", 'Amylase normal')|[('Gamma-glutamyltransferase normal', 1.0), ('Hepatic steatosis', 1.0), ('Ultrasound liver abnormal', 1.0), ('Blood alka

Nombre de patterns: 159672783
CPU times: user 1.25 s, sys: 327 ms, total: 1.58 s
Wall time: 8min 6s


### Résultats pour un symptôme

Après avoir exécuté le MBA pour la plus grande collection d'observations, sélectionnez et affichez les règles d'association (nom du symptôme et valeur d'association) du symptôme **d'ID 11320**, c'est-à-dire lorsque le symptôme est seul. La sortie doit être formatée dans un tableau, où chaque ligne contient les informations d'un symptôme associé.

Ce qui est attendu :
- Titre du tableau : ID du symptôme 11320 et son nom
- Nombre de lignes du tableau
- Tableau contenant 2 colonnes : Symptôme (Nom du symptôme associé) et Information (Niveau de confiance)
- **Important** : La première colonne (symptôme) doit être classée par ordre alphabétique

Signalez le temps d'exécution.

In [11]:
%%time
nom_11320 = df_symptoms.filter(df_symptoms.SYMPTOM_ID == 11320).first()['SYMPTOM_NAME']

# quand le symptome est seul la cle est un tuple a 1 element, genre ('Dizziness',)
res = assoc_rules.filter(lambda r: r[0] == (nom_11320,)).collect()
lignes = res[0][1] if len(res) > 0 else []

# tri alphabetique sur le nom du symptome associe
lignes = sorted(lignes, key=lambda x: x[0])

print("Symptôme ID 11320 :", nom_11320)
print("Nombre de lignes :", len(lignes))
df_11320 = spark.createDataFrame(lignes, "Symptome string, Information double")
# limiter l'affichage a 50 sinon c'est super long
df_11320.show(50, truncate=False)
# changer avec ça si on veut tout
# df_11320.show(max(len(lignes), 1), truncate=False)
df_11320.coalesce(1).write.mode("overwrite").csv(ROOT + "resultats_11320", header=True)

Symptôme ID 11320 : Dizziness
Nombre de lignes : 8611
+-------------------------------------+---------------------+
|Symptome                             |Information          |
+-------------------------------------+---------------------+
|5-alpha reductase inhibition therapy |5.9589428835324615E-6|
|ACTH stimulation test                |1.7876828650597385E-5|
|ACTH stimulation test abnormal       |1.1917885767064923E-5|
|ACTH stimulation test normal         |1.7876828650597385E-5|
|ADAMTS13 activity assay              |5.9589428835324615E-6|
|ADAMTS13 activity decreased          |1.1917885767064923E-5|
|ASIA syndrome                        |1.1917885767064923E-5|
|AST to platelet ratio index increased|5.9589428835324615E-6|
|AST/ALT ratio                        |2.9794714417662307E-5|
|AST/ALT ratio abnormal               |5.9589428835324615E-6|
|Abasia                               |9.891845186663886E-4 |
|Abdomen scan                         |7.150731460238954E-5 |
|Abdomen scan no

CPU times: user 69.9 ms, sys: 31.9 ms, total: 102 ms
Wall time: 14.2 s


Estimez l’empreinte carbone de votre exécution sur GCP à l’aide d’un calculateur d’empreinte carbone pour les workloads cloud. Indiquez les paramètres utilisés et le résultat obtenu (en kgCO₂e). Commentez brièvement votre résultat.

Vous pouvez utiliser le calculateur suivant: https://tools.finitizer.com/cloud-carbon-calculator, ou tout autre estimateur de votre choix.

J'ai calculé sur tools finitzer.

Paramètres utilisés
- Fournisseur : GCP, région "us-east4, Nothern Virginia"
- 3 VM "Another size" (1 master + 2 workers), 16 vCPU et 128 Go de RAM au total
- Durée : environ 5 h (temps où le cluster était allumé, pas seulement le temps du job)

Résultat : environ 0.5 à 1 kgCO2 équivalent, (le résultat était de 6 à 10, mais j'ai divisé
par 12 puisque le simulateur roule en quantité annuelle) ce qui équivaut à quelques kilomètres en voiture.

Nous avons remarqué que la région change énormément le résultat. Le calculateur donne environ 11 g/kWh pour Montréal,
qui roule presque juste à l’hydroélectricité, alors que us-east4 (Virginie du Nord) est bien plus élevée. Si on avait
lancé exactement le même job à Montréal, on aurait émis BEAUCOUP moins.

On a aussi vu que ce n’est pas le calcul lui-même qui pèse le plus. Notre MBA a pris seulement 8 minutes, mais le cluster
est resté allumé environ 5 h (création, configuration, tests, exécutions), et c’est ce temps-là qui compte pour l’empreinte.
Ça montre pourquoi il faut supprimer le cluster dès qu’on a fini, en plus d’économiser nos crédits. Il faut quand même 
garder en tête que notre estimation est plutôt un minimum : il manque la fabrication du matériel le stockage,
et l’autoscaling a peut-être ajouté un troisième worker pendant le job.
